# 🔀 Lab 04 — Put the diff on screen
Run Lab 01 first. The normal bundle run uses `channel_value=tv_app` and should pass. Re-run it with `channel_value=tv-app` to stage a configuration incident. For the Git exercise, use the separate Git-source job and deliberately commit a broken predicate on a disposable branch as described in the README.

Write down the independent acceptance criteria from Lab 01. Then run the job and record the run ID and source SHA (Git variant). Ask for evidence behind the proposed fix before requesting a review branch.


In [ ]:
%sql
SELECT assert_true(:catalog RLIKE '^[A-Za-z_][A-Za-z0-9_]*$' AND :schema RLIKE '^zeroops_lab_[a-z0-9_]+$', 'Use a dedicated zeroops_lab_<alias> schema');


In [ ]:
%sql
SELECT channel, currency, COUNT(*) AS purchases, SUM(amount_cents) AS cents
FROM IDENTIFIER(:catalog || '.' || :schema || '.zops_feed_v1')
GROUP BY channel, currency ORDER BY channel, currency;


## The contract gate
This gate expects two unique USD purchases and 2,000 cents. When the notebook is correct and the run parameter is wrong, fix the parameter and rerun. When the checked-out predicate itself is wrong, review a code correction that keeps the web purchase out and leaves the assertion in place. Both incidents can end with the exact same SQL exception.


In [ ]:
%sql
WITH tv_app_purchases AS (
  SELECT purchase_id, currency, amount_cents
  FROM IDENTIFIER(:catalog || '.' || :schema || '.zops_feed_v1')
  WHERE channel = :channel_value
)
SELECT COUNT(*) AS purchases, SUM(amount_cents) AS cents,
       assert_true(COUNT(*) = 2 AND COUNT(DISTINCT purchase_id) = 2
          AND COUNT_IF(currency = 'USD') = 2 AND SUM(amount_cents) = 2000,
          'TV app contract requires two unique USD purchases and 2000 cents') AS contract_check
FROM tv_app_purchases;


## Take two
For a paired comparison, use two copies of the job with the same code and bad parameters. Let the first investigation finish before adding the sample instructions, then fail the other job. Score both reports on evidence, diagnosis and action. Record whether the second report loaded the instructions. Two cases are a walkthrough, and a missing second report leaves the comparison incomplete.

On the Git path, keep the run input at `tv_app` on both revisions. Inspect the candidate diff and run a separate scratch job against the candidate commit. If Create PR is unavailable, prepare a manual review patch and record its origin. Track the branch push, PR, successful run and resolution as separate events.
